In [0]:
%sql
USE dbjahan.bronze_data;
CREATE OR REPLACE TABLE dbjahan.bronze_data.bilar_trafik_raw (
  region              STRING,
  region_text         STRING,
  agarkategori        STRING,
  agarkategori_text   STRING,
  tid                 STRING,
  antal               BIGINT,
  hamtad_tid          TIMESTAMP
)
USING DELTA;

In [0]:
%sql
DESCRIBE TABLE EXTENDED dbjahan.bronze_data.bilar_trafik_raw

col_name,data_type,comment
region,string,null
region_text,string,null
agarkategori,string,null
agarkategori_text,string,null
tid,string,null
antal,bigint,null
hamtad_tid,timestamp,null
,,
# Detailed Table Information,,
Catalog,dbjahan,


In [0]:
import requests
import itertools
import time
from datetime import datetime
from pyspark.sql import Row

def hamta(api_namn, url, params=None, timeout=90):
    senaste = None
    for forsok in range(1, 4):
        try:
            svar = requests.get(url, params=params, timeout=timeout)
            if svar.status_code == 200:
                return svar
            senaste = str(svar.status_code)
        except requests.RequestException as fel:
            senaste = str(fel)
        if forsok < 3:
            time.sleep(5)
    raise RuntimeError(
        f"{api_namn} svarade {senaste} efter 3 försök. URL: {url}"
    )

url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB3276/data"
params = {
    "lang": "sv",
    "outputFormat": "json-stat2",
    "valueCodes[ContentsCode]": "TK1001AB",
    "valueCodes[Region]": "*",
    "valueCodes[Agarkategori]": "*",
    "valueCodes[Tid]": "from(2020)",
}

svar = hamta("SCB TAB3276", url, params=params, timeout=90)
js = svar.json()

ids = js["id"]
dims = []
for var in ids:
    cat = js["dimension"][var]["category"]
    index = cat["index"]
    labels = cat.get("label", {})
    codes = [None] * len(index)
    for kod, plats in index.items():
        codes[plats] = (kod, labels.get(kod, kod))
    dims.append(codes)

nu = datetime.now()
rader = []
for i, combo in enumerate(itertools.product(*dims)):
    d = {var: kod for var, (kod, _txt) in zip(ids, combo)}
    t = {var: txt for var, (_kod, txt) in zip(ids, combo)}
    val = js["value"][i]
    rader.append(Row(
        region=d["Region"],
        region_text=t["Region"],
        agarkategori=d["Agarkategori"],
        agarkategori_text=t["Agarkategori"],
        tid=d["Tid"],
        antal=int(val) if val is not None else None,
        hamtad_tid=nu,
    ))

df = spark.createDataFrame(rader)
df.createOrReplaceTempView("trafik_nya")

spark.sql("""
INSERT INTO dbjahan.bronze_data.bilar_trafik_raw
SELECT n.region, n.region_text, n.agarkategori, n.agarkategori_text,
       n.tid, n.antal, n.hamtad_tid
FROM trafik_nya n
WHERE NOT EXISTS (
  SELECT 1
  FROM dbjahan.bronze_data.bilar_trafik_raw t
  WHERE t.region = n.region
    AND t.agarkategori = n.agarkategori
    AND t.tid = n.tid
)
""")
print("Klart trafik", df.count())

Klart trafik 13230


In [0]:
%sql
SELECT * FROM dbjahan.bronze_data.bilar_trafik_raw
LIMIT 20;

region,region_text,agarkategori,agarkategori_text,tid,antal,hamtad_tid
0163,Sollentuna,020,män,2023,15536,2026-09-28T12:36:30.884Z
0163,Sollentuna,020,män,2024,15572,2026-09-28T12:36:30.884Z
0163,Sollentuna,020,män,2025,15908,2026-09-28T12:36:30.884Z
0163,Sollentuna,030,juridiker,2020,10150,2026-09-28T12:36:30.884Z
0163,Sollentuna,030,juridiker,2021,10155,2026-09-28T12:36:30.884Z
0163,Sollentuna,030,juridiker,2022,12131,2026-09-28T12:36:30.884Z
0163,Sollentuna,030,juridiker,2023,12441,2026-09-28T12:36:30.884Z
0163,Sollentuna,030,juridiker,2024,12996,2026-09-28T12:36:30.884Z
0163,Sollentuna,030,juridiker,2025,15171,2026-09-28T12:36:30.884Z
0163,Sollentuna,040,taxi,2020,194,2026-09-28T12:36:30.884Z
